
###### 15_evaluation

###### Purpose

The purpose of this notebook is to evaluate Agentic RAG responses using both automatic keyword matching and LLM-as-a-Judge scoring to measure retrieval quality, relevance, groundedness, clarity, and hallucination risk.


###### Technologies Used

- Databricks

- Delta Lake

- Unity Catalog

- Databricks Vector Search

- Databricks Embedding Foundation Model (databricks-gte-large-en)

- LLM Model (databricks-meta-llama-3-1-8b-instruct)

- Python

- Databricks SDK


###### Input

- User questions

- Expected keywords

- Existing Vector Search index

- LLM endpoint

- Embedding endpoint

- Retrieved customer notes


######  Output

- Generated answer

- Keyword score

- LLM-as-Judge scores

   - Relevance
   - Groundedness
   - Clarity
   - Reason


######  Architecture

```text

Question
   ↓
Agentic RAG / RAG
   ↓
Retrieved Context
   ↓
Generated Answer
   ↓
Evaluation Metrics

```


###### Section 0 : Install Vector Search client

In [0]:
%pip install databricks-vectorsearch
dbutils.library.restartPython()

###### Section 1 :   Load Project Configuration

In [0]:
%run ./00_project_config

###### Section 2 : Import Libraries and Initialize Clients

In [0]:
from databricks.sdk import WorkspaceClient
from databricks.vector_search.client import VectorSearchClient
from databricks.sdk.service.serving import ChatMessage, ChatMessageRole

w = WorkspaceClient()
vsc = VectorSearchClient(disable_notice=True)

###### Section 3 : Connect to Existing Vector Search Index

In [0]:
index = vsc.get_index(
    endpoint_name = VECTOR_SEARCH_ENDPOINT_NAME,
    index_name = VECTOR_INDEX_NAME
)

index_description = index.describe()

print("Connected to the existing Vector Search index.")
print(
    "Index state:",
    index_description
    .get("status", {})
    .get("detailed_state", "UNKNOWN")
)

###### Section 4 : Define LLM Response Helper

In [0]:
def generate_answer(prompt: str):
    response = w.serving_endpoints.query(
        name=LLM_MODEL,
        messages=[
            ChatMessage(
                role=ChatMessageRole.USER,
                content=prompt
            )
        ],
        max_tokens=300,
        temperature=0.0
    )

    if (
        not response.choices
        or response.choices[0].message is None
    ):
        raise ValueError(
            "The LLM returned no response."
        )

    return response.choices[0].message.content

###### Section 5 : Define Vector Search Tool

In [0]:
def search_customer_notes(
    question: str,
    num_results: int = 3
):
    if not question or not question.strip():
        raise ValueError("Question cannot be empty.")

    if num_results <= 0:
        raise ValueError(
            "num_results must be greater than zero."
        )

    response = w.serving_endpoints.query(
        name=EMBEDDING_MODEL,
        input=[question]
    )

    if (
        not response.data
        or response.data[0].embedding is None
    ):
        raise ValueError(
            "The embedding model returned no embedding."
        )

    question_embedding = [
        float(value)
        for value in response.data[0].embedding
    ]

    results = index.similarity_search(
        query_vector=question_embedding,
        columns=["customer_id", "note"],
        num_results=num_results
    )

    rows = (
        results
        .get("result", {})
        .get("data_array", [])
    )

    if not rows:
        return {
            "tool": "search_customer_notes",
            "status": "no_results",
            "context": "",
            "rows": [],
            "result_count": 0
        }

    context_lines = [
        f"Customer {int(customer_id)}: {note}"
        for customer_id, note, score in rows
    ]

    return {
        "tool": "search_customer_notes",
        "status": "success",
        "context": "\n".join(context_lines),
        "rows": rows,
        "result_count": len(rows)
    }

###### Section 6 : Define SQL Analytics Tool

In [0]:
def count_customer_notes():
    count = spark.table(NOTES_TABLE).count()

    return {
        "tool": "count_customer_notes",
        "status": "success",
        "count": count
    }

###### Section 7 : Tool execution

In [0]:
VALID_TOOLS = {
    "count_customer_notes",
    "search_customer_notes"
}


def choose_tool(question: str) -> str:
    if not question or not question.strip():
        raise ValueError("Question cannot be empty.")

    tool_prompt = f"""
You are a tool-routing agent.

Choose the best tool for the user's question.

Available tools:

1. count_customer_notes
Use for questions about counts, totals, how many, or the number
of customer notes.

2. search_customer_notes
Use for questions about reasons, complaints, dissatisfaction,
cancellation, customer sentiment, or other semantic information
contained in customer notes.

Question:
{question}

Return exactly one tool name and nothing else:

count_customer_notes
or
search_customer_notes
"""

    tool_name = (
        generate_answer(tool_prompt)
        .strip()
        .lower()
        .replace("`", "")
        .replace('"', "")
        .replace("'", "")
        .replace(".", "")
        .strip()
    )

    if tool_name not in VALID_TOOLS:
        raise ValueError(
            f"LLM returned an unsupported tool: {tool_name}"
        )

    return tool_name

###### Section 8 : Agent

In [0]:
def agentic_rag_agent(question: str):
    if not question or not question.strip():
        raise ValueError("Question cannot be empty.")

    # Step 1: Let the LLM select the appropriate tool
    selected_tool = choose_tool(question)

    print(f"Tool selected by LLM: {selected_tool}")

    # Step 2: Execute the selected tool
    if selected_tool == "count_customer_notes":
        tool_result = count_customer_notes()

        if tool_result["status"] != "success":
            raise RuntimeError(
                f"Count tool failed: {tool_result}"
            )

        final_prompt = f"""
You are a telecom customer-support assistant.

Use only the tool result below to answer the user's question.
Do not add information that is not present in the tool result.

Question:
{question}

Tool used:
count_customer_notes

Tool result:
Customer-note count: {tool_result["count"]}

Provide a concise and clear final answer.
"""

    elif selected_tool == "search_customer_notes":
        tool_result = search_customer_notes(question)

        if tool_result["status"] == "no_results":
            return {
                "selected_tool": selected_tool,
                "tool_result": tool_result,
                "answer": (
                    "I don't have enough information from "
                    "the retrieved customer notes."
                )
            }

        if tool_result["status"] != "success":
            raise RuntimeError(
                f"Vector Search tool failed: {tool_result}"
            )

        context = tool_result["context"]

        final_prompt = f"""
You are a telecom customer-support assistant.

Answer the user's question using ONLY the retrieved customer notes.
Do not use outside knowledge or make assumptions.

If the retrieved notes do not contain enough information,
respond exactly:

"I don't have enough information from the retrieved customer notes."

Retrieved Customer Notes:
{context}

Question:
{question}

Provide a concise and factual answer.
"""

    else:
        raise ValueError(
            f"Unsupported selected tool: {selected_tool}"
        )

    # Step 3: Generate the final user-facing answer
    final_answer = generate_answer(final_prompt)

    # Step 4: Return a structured trace
    return {
        "selected_tool": selected_tool,
        "tool_result": tool_result,
        "answer": final_answer
    }

###### Section 9 : Evaluation Questions

In [0]:
eval_questions = [

{
"question":"How many customer notes are there?",
"expected_keywords":["8"]
},

{
"question":"Why are customers likely to cancel service?",
"expected_keywords":[
"cancel",
"terminate",
"service quality"
]
},

{
"question":"Billing complaints?",
"expected_keywords":[
"billing",
"charges",
"invoice"
]
},

{
"question":"Upgrade requests?",
"expected_keywords":[
"upgrade",
"plan"
]
},

{
"question":"Network issues?",
"expected_keywords":[
"internet",
"slow",
"disconnect"
]
},

{
"question":"Contract problems?",
"expected_keywords":[
"contract",
"termination"
]
}

]


###### Section 10 : Basic keyword evaluation

In [0]:
def keyword_score(answer, expected_keywords):
    answer_lower = answer.lower()
    matches = 0

    for keyword in expected_keywords:
        if keyword.lower() in answer_lower:
            matches += 1

    return matches / len(expected_keywords)

###### Section 11 : LLM-as-Judge Evaluation

In [0]:

def judge_answer(question, answer):
    judge_prompt = f"""
You are evaluating an AI assistant answer.

Question:
{question}

Answer:
{answer}

Score the answer from 1 to 5 for:
1. Relevance
2. Groundedness
3. Clarity
4. Reason
5. Hallucination Risk

Return only this format:

Relevance: <score>
Groundedness: <score>
Clarity: <score>
Hallucination Risk: <score>
Reason: <short reason>

"""

    try:
        return call_llm(judge_prompt)
    except Exception as e:
        return f"Evaluation failed: {e}"

###### Section 12 : Run Agent

In [0]:
#run agent
eval_results = []

for item in eval_questions:
    question = item["question"]
    response = agentic_rag_agent(question)
    answer = response["answer"]

    eval_results.append({
        "question": question,
        "answer": answer,
        "expected_keywords": item["expected_keywords"]
    })

display(spark.createDataFrame(eval_results))

scored_results = []

for row in eval_results:
    score = keyword_score(
        row["answer"],
        row["expected_keywords"]
    )

    scored_results.append({
        "question": row["question"],
        "answer": row["answer"],
        "keyword_score": score
    })

display(spark.createDataFrame(scored_results))

for row in eval_results:
    print("QUESTION:", row["question"])
    print(judge_answer(row["question"], row["answer"]))
    print("-" * 80)

###### Notebook Summary

- Get Configurations for Vector Search endpoint name,  Vector index name,  Embedding model name and LLM endpoint name.

- Connect to Vector Search Index.

- Implemented reusable components for 
    
    - Vector Search Tool
    - SQL Tool
    - Call LLM
    - Choose tool
    - Agent
    - Evaluation Questions
    - Basic keyword evaluation
    - LLM-as-Judge Evaluation

- Test Agent 

###### Key Learnings

- Used Databricks Vector Search to retrieve semantically similar historical customer notes.
- Learned how to evaluate Agentic RAG responses using both keyword-based evaluation and LLM-as-a-Judge evaluation.
- Understood that traditional Machine Learning evaluates predictions against ground-truth labels.
- Learned that GenAI systems are evaluated based on retrieval quality, answer relevance, groundedness, clarity, and hallucination risk.
- Understood how to determine whether a RAG or Agentic RAG response is reliable.
   - In Machine Learning, common evaluation metrics include:
        - Accuracy
        - Precision
        - Recall
        - F1-score
        - ROC-AUC
   - In GenAI applications, common evaluation metrics include:
        - Retrieval quality
        - Answer relevance
        - Groundedness
        - Hallucination risk
- Understood the following evaluation concepts:
    - Relevance → Did the response answer the user's question?
    - Groundedness → Is the response supported by the retrieved context?
    - Clarity → Is the response easy to understand?
    - Hallucination Risk → Did the model avoid generating unsupported information?

###### Notebook Conclusion

- This notebook evaluated Agentic RAG responses using both automatic keyword matching and LLM-as-a-Judge scoring. This demonstrated how GenAI systems are evaluated differently from traditional machine learning by measuring retrieval quality, groundedness, clarity, and answer relevance instead of only prediction accuracy.

###### Next Notebook

16_single_agent_customer_support

The purpose of this notebook is to build a Teco Churn Agentic AI application that intelligently selects and executes multiple AI tools (SQL Analytics,  Vector Search, and LLM reasoning) to answer scustomer-related questions and generate grounded responses.